# Lab7 — Solución (compacta)

In [ ]:
from pathlib import Path
import pandas as pd

_ORIGINAL_READ_CSV = pd.read_csv


def _find_local_csv(filename):
    direct = Path.cwd() / filename
    if direct.exists():
        return direct
    for candidate in Path.cwd().rglob(filename):
        return candidate
    return None


def _read_csv_local_first(source, *args, **kwargs):
    if isinstance(source, str) and source.startswith("http"):
        local_copy = _find_local_csv(Path(source).name)
        if local_copy is not None:
            return _ORIGINAL_READ_CSV(local_copy, *args, **kwargs)
    return _ORIGINAL_READ_CSV(source, *args, **kwargs)

pd.read_csv = _read_csv_local_first

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.options.display.max_columns = 50

In [30]:
# Cargar datos
url = 'https://raw.githubusercontent.com/hernansalinas/Curso_aprendizaje_estadistico/main/datasets/Sesion_07_housing.csv'
df = pd.read_csv(url)
df.shape

(20640, 10)



 **Tamaño del dataset:** 20640 observaciones y 10 columnas. Esto define el conjunto completo con el que trabajamos; la muestra de entrenamiento/test se obtiene por partición estratificada.

 **Columnas y tipos (resumen):** la mayor parte de las columnas son numéricas (por ejemplo `median_income`, `median_house_value`, `total_rooms`, etc.) y `ocean_proximity` es una variable categórica de tipo cadena. Mantener este detalle es importante porque condiciona las transformaciones (imputación/estandarización para numéricos, encoding para categóricos).

 **Valores faltantes:** la única variable con NA relevantes es `total_bedrooms` (207 valores faltantes). Decidimos imputar por la mediana en el pipeline numérico (`SimpleImputer(strategy='median')`), ya que la mediana es robusta frente a outliers y preserva la distribución central sin sesgar la media. No eliminamos filas para conservar información espacial y de ingreso.

 **Categoría `ocean_proximity`:** categorías observadas: ['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']. Estas categorías se codifican con `OneHotEncoder(handle_unknown='ignore')` en el pipeline para que el modelo pueda utilizarlas como variables indicadoras sin introducir ordinality indebida.

(Los números concretos y la lista de categorías se obtienen ejecutando las celdas de EDA en el notebook.)

In [31]:
# Chequeos rápidos
print(df.info())
print(df.isnull().sum())
print(df['ocean_proximity'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
None
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
d

In [32]:
# Estadísticas por categoría (compacto)
cols = ['housing_median_age','total_rooms','total_bedrooms','population','households','median_income','median_house_value']
df.groupby('ocean_proximity')[cols].mean().round(2)

,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
ocean_proximity,,,,,,,
<1H OCEAN,29.28,2628.34,546.54,1520.29,517.74,4.23,240084.29
INLAND,24.27,2717.74,533.88,1391.05,477.45,3.21,124805.39
ISLAND,42.40,1574.60,420.40,668.00,276.60,2.74,380440.00
NEAR BAY,37.73,2493.59,514.18,1230.32,488.62,4.17,259212.31
NEAR OCEAN,29.35,2583.70,538.62,1354.01,501.24,4.01,249433.98


**Promedios de median_house_value por ocean_proximity:**

- `<1H OCEAN`: 240,084.29 — zonas cercanas al océano pero no necesariamente frente a la playa muestran un valor medio alto, probablemente por demanda y ubicación.
- `INLAND`: 124,805.39 — las zonas del interior son notablemente más baratas en promedio; podría reflejar menor demanda o menor acceso a servicios/atractivos costeros.
- `ISLAND`: 380,440.00 — pocas observaciones pero con valores medianos muy altos; indica que las islas del conjunto son propiedades de alto valor.
- `NEAR BAY`: 259,212.31 — valores altos cerca de bahías, similar a zonas costeras premium.
- `NEAR OCEAN`: 249,433.98 — también elevado, consistente con la cercanía al mar.

Interpretación: la proximidad al mar/bahía está fuertemente correlacionada con un mayor `median_house_value`. Esto sugiere que `ocean_proximity` es una variable predictiva relevante para el precio y por eso la incluimos mediante `OneHotEncoder` en el pipeline.

In [33]:
# Preparación y partición estratificada
df['income_cat'] = pd.cut(df['median_income'], bins=[0.,1.5,3.0,4.5,6.,np.inf], labels=[1,2,3,4,5])
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(df, df['income_cat']):
    housing_train = df.loc[train_idx].drop(columns=['income_cat']).copy()
    housing_test  = df.loc[test_idx].drop(columns=['income_cat']).copy()
len(housing_train), len(housing_test)

(16512, 4128)

In [34]:
# Feature engineering + pipeline
for ds in (housing_train, housing_test):
    ds['rooms_per_household'] = ds['total_rooms'] / ds['households']
    ds['bedrooms_per_room'] = ds['total_bedrooms'] / ds['total_rooms']
    ds['population_per_household'] = ds['population'] / ds['households']
num_attribs = ['longitude','latitude','housing_median_age','total_rooms','total_bedrooms','population','households','median_income','rooms_per_household','bedrooms_per_room','population_per_household']
cat_attribs = ['ocean_proximity']
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])
full_pipeline = ColumnTransformer([('num', num_pipeline, num_attribs),('cat', OneHotEncoder(handle_unknown='ignore'), cat_attribs)])
X_train = full_pipeline.fit_transform(housing_train)
y_train = housing_train['median_house_value'].values
X_test = full_pipeline.transform(housing_test)
y_test = housing_test['median_house_value'].values
X_train.shape, X_test.shape

((16512, 16), (4128, 16))

In [35]:
# Modelo base y evaluación
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, preds))
mse = mean_squared_error(y_test, preds)
print('RMSE:', np.sqrt(mse))
print('R2:', r2_score(y_test, preds))

MAE: 48966.30850625835
RMSE: 66803.13890772291
R2: 0.6575916553584118


**Conclusiones del modelo :**

 **Métricas obtenidas :**
   MAE ≈ 48,966.31 — el error absoluto medio en las predicciones es ≈ $48.9k; en promedio la predicción difiere del valor real por ese monto.
   RMSE ≈ 66,803.14 — la raíz del error cuadrático medio es mayor que la MAE, lo que indica que existen errores grandes (outliers) que penalizan al RMSE; RMSE es más sensible a grandes desviaciones.
   R² ≈ 0.658 — el modelo lineal explica aproximadamente el 65.8% de la varianza del precio en el set de test.

 **Interpretación práctica:**
   Un MAE de ~$49k es relevante cuando los valores de las viviendas van desde ~ tens to cientos de miles; por tanto, aunque el modelo captura la tendencia general, su precisión absoluta deja espacio para mejoras, especialmente si buscamos predicciones finas para precios individuales.
   R² ~0.66 se considera una capacidad explicativa moderada-alta en problemas de precio inmobiliario con muchas fuentes de variabilidad no observadas; no obstante, no implica que las predicciones individuales sean siempre precisas.

 **Decisiones de modelado y por qué:**
   Usamos `ColumnTransformer` + `Pipeline` para asegurar que imputación, escalado y encoding se apliquen consistentemente durante entrenamiento y a nuevos datos. Imputamos `total_bedrooms` por mediana para manejar NAs sin eliminar observaciones.
   Estratificamos por `income_cat` (categoría derivada de `median_income`) al partir en train/test para mantener la misma distribución de ingreso relativo en ambos conjuntos y evitar sesgos en la evaluación del modelo.

 **Posibles mejoras y siguientes pasos recomendados:**
  1. Probar transformaciones del target (por ejemplo `log(median_house_value)`) si la distribución es sesgada; esto suele estabilizar errores y mejorar ajuste lineal.
  2. Validación cruzada y búsqueda de hiperparámetros con `GridSearchCV` o `RandomizedSearchCV` (incluir modelos con regularización: `Ridge`, `Lasso`).
  3. Evaluar modelos no lineales: `RandomForest`, `GradientBoosting` o `XGBoost`; suelen capturar interacciones y no-linealidades presentes en precios inmobiliarios.
  4. Ingeniería de características más rica: variables espaciales (cluster, distancias a servicios), interacción entre `median_income` y `ocean_proximity`, uso de polinomios o splines para `median_income`/`housing_median_age`.
  5. Análisis de residuos para detectar heterocedasticidad o zonas donde el modelo falla sistemáticamente; usar esa información para diseñar características o modelos locales.
  6. Si se busca interpretación, revisar los coeficientes del modelo lineal (tras escalado) o usar técnicas de explicación como SHAP para modelos complejos.

- **Resumen:** el pipeline y el modelo lineal sirven como una línea base sólida que explica gran parte de la variación del precio, pero para reducir errores absolutos (MAE/RMSE) y capturar no linealidades, conviene explorar transformaciones del target, regularización, y modelos basados en árboles con CV.